# ⚡ Kilat: General LLM Pre-Training in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Airukua/Kilat/blob/main/notebooks/kilat_general_pretraining_colab.ipynb)

```
  ██╗  ██╗██╗██╗      █████╗ ████████╗
  ██║ ██╔╝██║██║     ██╔══██╗╚══██╔══╝
  █████╔╝ ██║██║     ███████║   ██║   
  ██╔═██╗ ██║██║     ██╔══██║   ██║   
  ██║  ██╗██║███████╗██║  ██║   ██║   
  ╚═╝  ╚═╝╚═╝╚══════╝╚═╝  ╚═╝   ╚═╝   
  ────────────────────────────────────────────────────────────
  ⚡ KILAT | Kernelized Lightweight Attention Transformer (v2.0.0)
  Architecture: Hybrid Sub-Quadratic Global Decay + Latent MLA
  ────────────────────────────────────────────────────────────
```

This notebook demonstrates how to pre-train a causal language model from scratch using the **Kilat Python API** on GPU accelerators.

### Core Architectural Pillars:
- **Hybrid Sub-Quadratic Global Decay**: $O(N)$ linear-time attention for long-context associative recall.
- **Multi-Head Latent Attention (MLA)**: Low-rank latent key/value compression for ultra-efficient KV caching.
- **SwiGLU Mixture-of-Experts (MoE)**: Fine-grained dynamic expert routing with balanced auxiliary loss.
- **Rotary Positional Embeddings (RoPE)**: High-fidelity relative positional extrapolation.

## 1. Hardware & Accelerator Verification
Ensure your Google Colab runtime is set to **GPU** (*Runtime -> Change runtime type -> T4 / A100 / L4 GPU*).

In [ ]:
# ==============================================================================
#   ██╗  ██╗██╗██╗      █████╗ ████████╗
#   ██║ ██╔╝██║██║     ██╔══██╗╚══██╔══╝
#   █████╔╝ ██║██║     ███████║   ██║   
#   ██╔═██╗ ██║██║     ██╔══██║   ██║   
#   ██║  ██╗██║███████╗██║  ██║   ██║   
#   ╚═╝  ╚═╝╚═╝╚══════╝╚═╝  ╚═╝   ╚═╝   
#   ────────────────────────────────────────────────────────────
#   ⚡ KILAT | General LLM Pre-Training Pipeline (Google Colab)
#   ────────────────────────────────────────────────────────────
# ==============================================================================
# Check NVIDIA GPU acceleration details
!nvidia-smi

## 2. Clone Repository & Install Kilat

In [ ]:
# ==============================================================================
# ⚡ KILAT: Repository Installation & Dependency Setup
# ==============================================================================
!git clone https://github.com/Airukua/Kilat.git
%cd Kilat

# Install Kilat in editable mode along with production ML libraries
!pip install -e .
!pip install datasets transformers triton safetensors accelerate

## 3. Tokenizer Setup
Initialize any standard Byte-Pair Encoding (BPE) or WordPiece tokenizer (e.g., `gpt2` with 50,257 vocab).

In [ ]:
# ==============================================================================
# ⚡ KILAT: Tokenizer Initialization
# ==============================================================================
from transformers import AutoTokenizer

# Load standard GPT-2 BPE tokenizer
tokenizer_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

# Enforce pad token definition for causal padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"[KILAT] Tokenizer:      '{tokenizer_name}'")
print(f"[KILAT] Vocabulary Size: {len(tokenizer):,}")
print(f"[KILAT] Pad Token ID:    {tokenizer.pad_token_id} ({tokenizer.pad_token})")
print(f"[KILAT] EOS Token ID:    {tokenizer.eos_token_id} ({tokenizer.eos_token})")

## 4. General Pre-Training Corpus Ingestion
Load any general-purpose pre-training text corpus (e.g. `wikitext`, `HuggingFaceFW/fineweb-edu`, or your domain-specific text dataset) and wrap it with `KilatDataCollator`.

`KilatDataCollator` guarantees:
- Autoregressive sequence alignment (`input_ids` and `labels`).
- Masked padding tokens (`-100` ignore index) to eliminate training loss contamination.
- Zero-copy tensor collation fast paths.

In [ ]:
# ==============================================================================
# ⚡ KILAT: General Corpus Ingestion & Causal Data Pipeline
# ==============================================================================
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from kilat.data.collator import KilatDataCollator

# Ingest a general text pre-training corpus
# (You can replace 'wikitext' with 'HuggingFaceFW/fineweb-edu', 'c4', or local files)
print("[KILAT] Downloading pre-training corpus from Hugging Face...")
raw_train = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
raw_eval = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")

class GeneralPretrainingDataset(Dataset):
    """
    Generic PyTorch Dataset for causal language modeling pre-training.
    Yields dictionary samples with 'input_ids' for KilatDataCollator.
    """
    def __init__(self, hf_dataset, tokenizer, max_length=512, text_column="text"):
        # Filter out empty or whitespace-only lines
        self.texts = [t for t in hf_dataset[text_column] if t.strip()]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            return_tensors=None,
        )
        return {"input_ids": encoded["input_ids"]}

MAX_SEQ_LENGTH = 512
train_dataset = GeneralPretrainingDataset(raw_train, tokenizer, max_length=MAX_SEQ_LENGTH)
eval_dataset = GeneralPretrainingDataset(raw_eval, tokenizer, max_length=MAX_SEQ_LENGTH)

# Kilat Data Collator handles causal padding and labels preparation
collator = KilatDataCollator(
    pad_token_id=tokenizer.pad_token_id,
    max_length=MAX_SEQ_LENGTH,
    ignore_index=-100,
    padding_side="right",
)

BATCH_SIZE = 8
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    drop_last=True,
    pin_memory=torch.cuda.is_available(),
)
eval_loader = DataLoader(
    eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    drop_last=True,
    pin_memory=torch.cuda.is_available(),
)

print(f"[KILAT] Dataset initialized!")
print(f"[KILAT] Training Batches:   {len(train_loader):,}")
print(f"[KILAT] Validation Batches: {len(eval_loader):,}")

## 5. Model Architecture Specification
Define a Kilat model configuration with `KilatConfig`.

Configurable architectural hyperparameters:
- `n_layer`, `n_embd`, `n_head`: Depth and width scaling.
- `recall_ratio`: Proportion of Multi-Head Latent Attention (MLA) heads vs. Sub-Quadratic Global Decay heads.
- `latent_dim`: Low-rank KV compression dimension.
- `num_experts`, `active_experts`: Fine-grained SwiGLU Mixture-of-Experts routing.

In [ ]:
# ==============================================================================
# ⚡ KILAT: Model Architecture Configuration & Initialization
# ==============================================================================
from kilat.configs import KilatConfig
from kilat.models import KilatTransformer

# Configure a 35M-parameter Kilat Hybrid Transformer
config = KilatConfig(
    vocab_size=len(tokenizer),
    max_position_embeddings=MAX_SEQ_LENGTH,
    n_embd=384,                       # Hidden embedding dimension
    n_layer=8,                        # Number of transformer blocks
    n_head=6,                         # Number of attention heads
    recall_ratio=0.5,                 # 50% Latent MLA heads, 50% Sub-Quadratic Global Decay
    latent_dim=96,                    # KV compression latent dimension
    use_rope=True,                    # Rotary Position Embeddings
    rope_head_dim=32,
    rope_base=10000.0,
    ffn_mode="swiglu",                # SwiGLU Feed-Forward Network
    ff_mult=2.6666666667,             # SwiGLU dimension multiplier
    num_experts=8,                    # Mixture-of-Experts: 8 routed experts
    active_experts=2,                 # Top-2 dynamic routing
    aux_loss_coef=0.01,               # Expert load balancing coefficient
    tie_word_embeddings=True,         # Tie input embeddings with output LM head
)

model = KilatTransformer(config)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"[KILAT] Model Architecture:   {config.model_type}")
print(f"[KILAT] Total Parameters:       {total_params:,}")
print(f"[KILAT] Trainable Parameters:   {trainable_params:,}")

## 6. Pre-Training Loop with `KilatTrainer`
Set up typed `TrainingArguments` and start pre-training.

### Kilat Optimization Sanity Invariants:
- Automatic mixed precision detection (`bf16` on Ampere/Ada/Hopper+, `fp16` on T4/V100).
- Weight decay parameter separation (biases, 1D layer norms, and decay parameters are excluded from decay).
- Partial accumulation loss scaling.
- Pure token-weighted cross-entropy evaluation (perplexity uncontaminated by router auxiliary loss).

In [ ]:
# ==============================================================================
# ⚡ KILAT: Training Arguments & KilatTrainer Pre-Training Loop
# ==============================================================================
from kilat.engine import KilatTrainer, TrainingArguments

# Automatically detect optimal mixed precision format
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    precision_mode = "bf16"
elif torch.cuda.is_available():
    precision_mode = "fp16"
else:
    precision_mode = "fp32"

training_args = TrainingArguments(
    output_dir="./checkpoints_general_pretrain",
    training_mode="steps",
    max_steps=1000,                  # Set according to compute budget (e.g. 5,000 - 50,000)
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,   # Effective batch size: 8 * 4 = 32
    learning_rate=6e-4,              # Peak learning rate
    weight_decay=0.01,
    warmup_steps=100,
    scheduler_type="cosine",
    precision=precision_mode,
    logging_steps=25,
    eval_steps=200,
    save_steps=500,
    save_checkpoints=True,
    save_total_limit=2,
    seed=42,
)

trainer = KilatTrainer(
    model=model,
    args=training_args,
    train_dataloader=train_loader,
    eval_dataloader=eval_loader,
    tokenizer=tokenizer,
)

print(f"[KILAT] Launching pre-training on device: {trainer.device} (precision: {precision_mode})...")
train_state = trainer.train()
print(f"\n[KILAT] Pre-training completed successfully at global step {train_state.global_step}!")

## 7. Autoregressive Text Generation & Evaluation
Evaluate generation quality using Kilat's native `TextGenerator` with KV-caching and nucleus sampling.

In [ ]:
# ==============================================================================
# ⚡ KILAT: Text Generation & Inference
# ==============================================================================
from kilat.inference.generator import TextGenerator

device = "cuda" if torch.cuda.is_available() else "cpu"
generator = TextGenerator(model=model, tokenizer=tokenizer, device=device)

test_prompts = [
    "Artificial intelligence has revolutionized",
    "In the early history of science,",
    "The development of efficient neural architectures"
]

print("=" * 65)
print("⚡ KILAT AUTOREGRESSIVE GENERATION")
print("=" * 65)

for prompt in test_prompts:
    print(f"\n[PROMPT]: {prompt}")
    generated_text = generator.generate(
        prompt=prompt,
        max_new_tokens=64,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.1,
    )
    print(f"[GENERATED]:\n{generated_text}\n" + "-" * 65)

## 8. Export Model & Publish to Hugging Face Hub
Export model weights to standard Hugging Face format (`model.safetensors` + `config.json`) and push to Hugging Face Hub.

In [ ]:
# ==============================================================================
# ⚡ KILAT: Model Serialization & Hugging Face Hub Export
# ==============================================================================
import os

export_directory = "./exported_kilat_model"
os.makedirs(export_directory, exist_ok=True)

# Save model weights in safe_serialization format (safetensors)
model.save_pretrained(export_directory, safe_serialization=True)

# Save tokenizer configuration
tokenizer.save_pretrained(export_directory)

print(f"[KILAT] Model successfully exported to: '{export_directory}'")
print(f"[KILAT] Exported files: {os.listdir(export_directory)}")

# ------------------------------------------------------------------------------
# To push to Hugging Face Hub, uncomment and provide your write token:
# ------------------------------------------------------------------------------
# from huggingface_hub import login, HfApi
# login()  # Enter your Hugging Face write token
# api = HfApi()
# api.upload_folder(
#     folder_path=export_directory,
#     repo_id="your-username/kilat-35m-pretrained",
#     repo_type="model",
# )
# print("[KILAT] Successfully uploaded to Hugging Face Hub!")